# 7. Os números da apresentação

Este caderno repete as contas dos cadernos 01 e 02, com o mesmo código, e grava em
`src/content/ml/apresentacao.json` os números que os slides da AV1 desenham na rota `/ml` do site:
histogramas, caixas, correlações, a dispersão das seis linhas fora da regra e a EDA das features
novas.

Ele acrescenta **algumas conferências** que 01 e 02 não imprimem, e que os slides usam. Estão
marcadas com `# conferência nova` no código:

- nas seis linhas fora da regra, o peso que a planilha deu ao indicador 3 nos pontos e a razão
  entre a soma dos pontos e o resultado lançado;
- a correlação das colunas cruas também nas 190 unidades modeladas, sem as seis linhas;
- quantas colunas misturam `%` dentro das próprias unidades, e quantas só nas linhas de distrito;
- as vírgulas de milhar, as colunas de pontos e as linhas de distrito por distrito;
- as classes também nas 196 unidades, que é a base do caderno 01.

Os gráficos do site são desenhados a partir deste arquivo, na identidade visual do projeto, em
vez de colar a imagem do matplotlib. Para o número do slide não se separar do número do
caderno, `src/content/apresentacao-ml.test.ts` confere o JSON contra as **saídas impressas**
dos cadernos 01 e 02. Mudou a análise lá, rode este caderno de novo.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

df = pd.read_csv('../data/base nova completa.csv', encoding='cp1252')


def para_numero(coluna):
    texto = coluna.astype(str).str.strip().str.replace(',', '')
    valor = pd.to_numeric(texto.str.rstrip('%'), errors='coerce')
    return valor / texto.str.endswith('%').map({True: 100, False: 1})


def arredondar(valor, casas=4):
    return None if pd.isna(valor) else round(float(valor), casas)


convertida = df.copy()
convertida[df.columns[2:]] = df[df.columns[2:]].apply(para_numero)
eh_unidade = df['Tipo de unidade'] != 'DS'
numericas = convertida.select_dtypes(include='number').columns.tolist()
print(df.shape, '| numéricas depois da conversão:', len(numericas))

(244, 260) | numéricas depois da conversão: 258


## Etapa 2: o dataset

Tamanho, tipos na leitura crua e depois da conversão, o papel de cada coluna e o funil de
linhas até a base modelada.

In [2]:
def tipo_na_leitura(dtype):
    # o pandas 2 lê texto como `object` e o 3 como `str`; os dois contam como texto
    if pd.api.types.is_integer_dtype(dtype):
        return 'inteiro'
    if pd.api.types.is_float_dtype(dtype):
        return 'decimal'
    return 'texto'


tipos_crus = df.dtypes.map(tipo_na_leitura).value_counts()
papel = pd.Series(numericas).str.split(' — ').str[0].value_counts()

eda = pd.DataFrame({
    'tipo': df['Tipo de unidade'],
    'distrito': df['Distrito Sanitário'],
    'ind1': para_numero(df['Desempenho consolidado — Indicador 1']),
    'ind2': para_numero(df['Desempenho consolidado — Indicador 2']),
    'ind3': para_numero(df['Desempenho consolidado — Indicador 3']),
    'ind4': para_numero(df['Desempenho consolidado — Indicador 4']) / 0.4,
    'geral': para_numero(df['Desempenho geral']),
})
eda['familia'] = eda['tipo'].str.split().str[0]
unidades = eda[eda['tipo'] != 'DS'].reset_index(drop=True)

esperado = 0.2 * unidades['ind1'] + 0.2 * unidades['ind2'] + 0.2 * unidades['ind3'] + 0.4 * unidades['ind4']
fora_da_regra = (unidades['geral'] - esperado).abs() > 0.01
modeladas = unidades[~fora_da_regra].reset_index(drop=True)

base = {
    'linhas': int(df.shape[0]),
    'colunas': int(df.shape[1]),
    'celulas': int(df.shape[0] * df.shape[1]),
    'tipos_na_leitura': {k: int(v) for k, v in tipos_crus.items()},
    'categoricas': [
        {'coluna': 'Tipo de unidade', 'valores': int(df['Tipo de unidade'].nunique())},
        {'coluna': 'Distrito Sanitário', 'valores': int(df['Distrito Sanitário'].nunique())},
    ],
    'numericas': len(numericas),
    'papeis': [{'papel': p, 'colunas': int(n)} for p, n in papel.items()],
    'linhas_distrito': int((~eh_unidade).sum()),
    'unidades': int(len(unidades)),
    'fora_da_regra': int(fora_da_regra.sum()),
    'modeladas': int(len(modeladas)),
    'tipos_nas_modeladas': int(modeladas['tipo'].nunique()),
    'familias_nas_modeladas': sorted(modeladas['familia'].unique().tolist()),
}

# as seis linhas fora da regra são todas as unidades de alguns tipos? (conferência nova)
familias_fora = sorted(unidades.loc[fora_da_regra, 'familia'].unique().tolist())
base['familias_fora_da_regra'] = familias_fora
base['unidades_dessas_familias'] = int(unidades['familia'].isin(familias_fora).sum())
base['linhas_distrito_por_distrito'] = sorted(set(df.loc[~eh_unidade, 'Distrito Sanitário'].value_counts().tolist()))

abaixo = int((modeladas['geral'] < 0.9).sum())
classes = {'abaixo_de_90': abaixo, 'noventa_ou_mais': int(len(modeladas) - abaixo)}
abaixo_196 = int((unidades['geral'] < 0.9).sum())
classes_196 = {'abaixo_de_90': abaixo_196, 'noventa_ou_mais': int(len(unidades) - abaixo_196)}
print(json.dumps(base, ensure_ascii=False, indent=1))
print(classes, classes_196)

{
 "linhas": 244,
 "colunas": 260,
 "celulas": 63440,
 "tipos_na_leitura": {
  "texto": 201,
  "inteiro": 48,
  "decimal": 11
 },
 "categoricas": [
  {
   "coluna": "Tipo de unidade",
   "valores": 20
  },
  {
   "coluna": "Distrito Sanitário",
   "valores": 8
  }
 ],
 "numericas": 258,
 "papeis": [
  {
   "papel": "Meta",
   "colunas": 67
  },
  {
   "papel": "Desempenho",
   "colunas": 60
  },
  {
   "papel": "Quantidade atendida",
   "colunas": 40
  },
  {
   "papel": "Total avaliado",
   "colunas": 38
  },
  {
   "papel": "Valor informado",
   "colunas": 28
  },
  {
   "papel": "Desempenho consolidado",
   "colunas": 24
  },
  {
   "papel": "Desempenho geral",
   "colunas": 1
  }
 ],
 "linhas_distrito": 48,
 "unidades": 196,
 "fora_da_regra": 6,
 "modeladas": 190,
 "tipos_nas_modeladas": 17,
 "familias_nas_modeladas": [
  "CAPS",
  "CECO",
  "MAC",
  "UBT",
  "UCIS",
  "USF"
 ],
 "familias_fora_da_regra": [
  "NDI",
  "SAE"
 ],
 "unidades_dessas_familias": 6,
 "linhas_distrito_por_

## Etapa 3: distribuição, caixas e correlação

Sobre as 196 unidades, como no caderno 01. Os histogramas usam os mesmos 25 intervalos do
`sns.histplot(bins=25)` de lá.

In [3]:
colunas = ['ind1', 'ind2', 'ind3', 'ind4', 'geral']


def histograma(serie, intervalos):
    contagens, bordas = np.histogram(serie.dropna(), bins=intervalos)
    return {'contagens': contagens.tolist(), 'de': arredondar(bordas[0]), 'ate': arredondar(bordas[-1])}


distribuicao = []
for coluna in colunas:
    valores = unidades[coluna]
    degraus = valores.round(4).value_counts().head(2)
    distribuicao.append({
        'coluna': coluna,
        'histograma': histograma(valores, 25),
        'assimetria': arredondar(valores.skew(), 2),
        'valores_distintos': int(valores.round(4).nunique()),
        'degraus': [{'valor': arredondar(v), 'unidades': int(n)} for v, n in degraus.items()],
        'minimo': arredondar(valores.min()),
        'maximo': arredondar(valores.max()),
    })


def caixa(valores):
    # as mesmas estatísticas do boxplot do seaborn: quartis, bigodes a 1,5 IQR e pontos fora
    valores = valores.dropna()
    q1, mediana, q3 = valores.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    dentro = valores[(valores >= q1 - 1.5 * iqr) & (valores <= q3 + 1.5 * iqr)]
    fora = valores[(valores < q1 - 1.5 * iqr) | (valores > q3 + 1.5 * iqr)]
    return {
        'n': int(len(valores)),
        'q1': arredondar(q1), 'mediana': arredondar(mediana), 'q3': arredondar(q3),
        'bigode_baixo': arredondar(dentro.min()), 'bigode_alto': arredondar(dentro.max()),
        'fora': sorted(arredondar(v) for v in fora),
    }


ordem_familias = unidades['familia'].value_counts().index.tolist()
por_familia = [{'grupo': f, **caixa(unidades.loc[unidades['familia'] == f, 'geral'])} for f in ordem_familias]
ordem_distritos = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII']
por_distrito = [{'grupo': d, **caixa(unidades.loc[unidades['distrito'] == d, 'geral'])} for d in ordem_distritos]

matriz = unidades[colunas].corr()
correlacao = {'colunas': colunas, 'matriz': [[arredondar(v, 4) for v in linha] for linha in matriz.values]}

brutas = convertida.loc[eh_unidade, numericas]
brutas = brutas.loc[:, brutas.nunique() > 1]
correlacao_geral = brutas.drop(columns='Desempenho geral').corrwith(brutas['Desempenho geral'])
mais_correlatas = correlacao_geral[correlacao_geral.abs().sort_values(ascending=False).index].head(15)
base['colunas_correlacionadas'] = int(len(correlacao_geral))

# as mesmas colunas nas 190 modeladas, sem as seis linhas fora da regra (conferência nova).
# coluna que fica constante sem elas não tem correlação definida: vai como null
sem_as_seis = convertida.loc[eh_unidade, numericas].reset_index(drop=True)[~fora_da_regra]
nas_modeladas = {
    c: (sem_as_seis[c].corr(sem_as_seis['Desempenho geral']) if sem_as_seis[c].nunique() > 1 else None)
    for c in mais_correlatas.index
}
assimetria_geral_modeladas = arredondar(modeladas['geral'].skew(), 2)

print(pd.DataFrame([{k: d[k] for k in ['coluna', 'assimetria', 'valores_distintos', 'degraus']} for d in distribuicao]))
print(matriz.round(2))
print(mais_correlatas.round(3))

  coluna  assimetria  valores_distintos  \
0   ind1       -0.38                  8   
1   ind2       -6.07                  2   
2   ind3        1.31                  8   
3   ind4        2.31                 45   
4  geral        3.84                 54   

                                             degraus  
0  [{'valor': 0.5, 'unidades': 108}, {'valor': 0....  
1  [{'valor': 0.5, 'unidades': 191}, {'valor': 0....  
2  [{'valor': 1.89, 'unidades': 171}, {'valor': 3...  
3  [{'valor': 0.8, 'unidades': 132}, {'valor': 0....  
4  [{'valor': 0.8974, 'unidades': 72}, {'valor': ...  
       ind1  ind2  ind3  ind4  geral
ind1   1.00  0.06  0.19  0.16   0.36
ind2   0.06  1.00  0.11 -0.04   0.16
ind3   0.19  0.11  1.00  0.36   0.79
ind4   0.16 -0.04  0.36  1.00   0.36
geral  0.36  0.16  0.79  0.36   1.00
Desempenho consolidado — Indicador 3.1        0.921
Desempenho consolidado — Indicador 3          0.791
Desempenho consolidado — Subindicador 4.IX   -0.757
Desempenho consolidado — Indicado

## Etapa 3: ausentes, outliers e inconsistências

In [4]:
faltantes = {
    'na_leitura': int(df.isna().sum().sum()),
    'depois_da_conversao': int(convertida.isna().sum().sum()),
    'colunas_com_ausente': int(convertida.isna().any().sum()),
}

q1, q3 = unidades[colunas].quantile(0.25), unidades[colunas].quantile(0.75)
fora_iqr = (unidades[colunas] < q1 - 1.5 * (q3 - q1)) | (unidades[colunas] > q3 + 1.5 * (q3 - q1))
outliers = [
    {'coluna': c, 'unidades': int(fora_iqr[c].sum()), 'percentual': arredondar(fora_iqr[c].mean() * 100, 1)}
    for c in colunas
]
outliers_geral_por_familia = unidades.loc[fora_iqr['geral'], 'familia'].value_counts()

constantes = convertida.columns[convertida[eh_unidade].nunique() <= 1]
mistura = [c for c in df.columns[2:] if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]
mistura = [c for c in mistura if 0 < df[c].astype(str).str.endswith('%').mean() < 1]
# quantas misturam dentro das próprias unidades; o resto só mistura por causa das linhas DS (conferência nova)
mistura_nas_unidades = [c for c in mistura if 0 < df.loc[eh_unidade, c].astype(str).str.endswith('%').mean() < 1]
como_texto = [c for c in df.columns[2:] if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]
depois_do_geral = df.columns[df.columns.get_loc('Desempenho geral') + 1:]
metas_duplas = [
    {'coluna': c, 'escritas': df[c].astype(str).unique().tolist(), 'valor': arredondar(convertida[c].iloc[0])}
    for c in df.columns if c.startswith('Meta') and df[c].nunique() > 1
]
com_virgula = df.iloc[:, 2:].astype(str).apply(lambda c: c.str.contains(','))
linhas_com_virgula = df.loc[com_virgula.any(axis=1), 'Tipo de unidade'].value_counts()

inconsistencias = {
    'colunas_constantes': int(len(constantes)),
    'colunas_fracao_e_percentual': int(len(mistura)),
    'mistura_dentro_das_unidades': int(len(mistura_nas_unidades)),
    'colunas_numericas_como_texto': int(len(como_texto)),
    'metas_escritas_de_dois_jeitos': metas_duplas,
    'celulas_com_virgula_de_milhar': int(com_virgula.sum().sum()),
    'linhas_com_virgula_de_milhar': {k: int(v) for k, v in linhas_com_virgula.items()},
    'colunas_de_pontos': int(len(depois_do_geral)),
    'pontos_com_nome_repetido': int(sum(str(c).endswith('.1') for c in depois_do_geral)),
}

print(faltantes)
print(pd.DataFrame(outliers))
print(outliers_geral_por_familia)
print(json.dumps(inconsistencias, ensure_ascii=False))

{'na_leitura': 0, 'depois_da_conversao': 0, 'colunas_com_ausente': 0}
  coluna  unidades  percentual
0   ind1         2         1.0
1   ind2         5         2.6
2   ind3        25        12.8
3   ind4        35        17.9
4  geral        29        14.8
familia
MAC     18
SAE      3
NDI      3
USF      3
UCIS     1
UBT      1
Name: count, dtype: int64
{"colunas_constantes": 96, "colunas_fracao_e_percentual": 49, "mistura_dentro_das_unidades": 6, "colunas_numericas_como_texto": 199, "metas_escritas_de_dois_jeitos": [{"coluna": "Meta — Subindicador 4.II.2", "escritas": ["100%", "1.00"], "valor": 1.0}, {"coluna": "Meta — Subindicador 4.IX.2", "escritas": ["20", "2000%"], "valor": 20.0}, {"coluna": "Meta — Subindicador 4.IX.3", "escritas": ["4", "400%"], "valor": 4.0}, {"coluna": "Meta — Subindicador 4.IX.3.1", "escritas": ["8", "800%"], "valor": 8.0}], "celulas_com_virgula_de_milhar": 114, "linhas_com_virgula_de_milhar": {"DS": 24}, "colunas_de_pontos": 4, "pontos_com_nome_repetido": 3}

## Etapa 3: as seis linhas fora da regra

O resultado geral de cada unidade deveria ser `0,2 ind1 + 0,2 ind2 + 0,2 ind3 + 0,4 ind4`.
Em seis linhas não é. Aqui fica o ponto de cada unidade (esperado pela portaria contra o
lançado na planilha) e, nas seis, os pontos que a planilha deu ao indicador 3.

In [5]:
pontos_ind3 = convertida.loc[eh_unidade, 'Desempenho consolidado — Indicador 3.1'].reset_index(drop=True)
peso_ind3 = (pontos_ind3 / unidades['ind3']).round(2)

# a soma dos quatro pontos dividida pelo resultado lançado (conferência nova): 1 nas unidades
# que seguem a portaria
colunas_de_pontos = [c for c in df.columns[df.columns.get_loc('Desempenho geral') + 1:]]
soma_dos_pontos = convertida.loc[eh_unidade, colunas_de_pontos].reset_index(drop=True).sum(axis=1)
razao = soma_dos_pontos / unidades['geral']
print('razão nas que seguem a regra:', razao[~fora_da_regra].round(3).unique().tolist())

dispersao = [
    {'esperado': arredondar(e), 'lancado': arredondar(g), 'fora': bool(f)}
    for e, g, f in zip(esperado, unidades['geral'], fora_da_regra)
]
linhas_fora = [
    {
        'tipo': unidades.loc[i, 'tipo'],
        'distrito': unidades.loc[i, 'distrito'],
        'esperado': arredondar(esperado[i]),
        'lancado': arredondar(unidades.loc[i, 'geral']),
        'peso_do_ind3_nos_pontos': arredondar(peso_ind3[i], 2),
        'soma_dos_pontos_sobre_lancado': arredondar(razao[i], 2),
    }
    for i in unidades.index[fora_da_regra]
]
pd.DataFrame(linhas_fora)

razão nas que seguem a regra: [1.0]


,tipo,distrito,esperado,lancado,peso_do_ind3_nos_pontos,soma_dos_pontos_sobre_lancado
0,SAE,I,1.3876,0.8161,0.2,1.7
1,SAE,III,1.3732,1.9049,0.8,1.7
2,SAE,IV,1.3291,1.8790,0.8,1.7
3,NDI,III,1.1816,1.7923,0.8,1.7
4,NDI,VI,1.1930,1.7989,0.8,1.7
5,NDI,IV,1.1356,1.7652,0.8,1.7


## Etapa 5: as features novas e a EDA delas

As mesmas seis features do caderno 02, sobre as 190 unidades modeladas.

In [6]:
origem = convertida[eh_unidade].reset_index(drop=True)[~fora_da_regra].reset_index(drop=True)
assert (origem['Desempenho consolidado — Indicador 1'] == modeladas['ind1']).all()

notas = ['ind1', 'ind2', 'ind3', 'ind4']
novas = pd.DataFrame(index=modeladas.index)
novas['taxa_ind1'] = origem['Quantidade atendida — Indicador 1'] / origem['Total avaliado — Indicador 1']
novas['ind3_x_ind4'] = modeladas['ind3'] * modeladas['ind4']
novas['ind3_abaixo_meta'] = (modeladas['ind3'] < 1).astype(int)
novas['indicadores_na_meta'] = (modeladas[notas] >= 1).sum(axis=1)
novas['desvio_indicadores'] = modeladas[notas].std(axis=1)
blocos_ind4 = [f'Desempenho consolidado — Subindicador 4.{b}' for b in ['I', 'II', 'III', 'IV', 'V', 'VII', 'VIII', 'X']]
novas['media_blocos_ind4'] = origem[blocos_ind4].mean(axis=1)

alvo = (modeladas['geral'] < 0.9).astype(int)
comparacao = pd.concat([modeladas[notas], novas], axis=1)
com_geral = comparacao.corrwith(modeladas['geral'])
com_abaixo = comparacao.corrwith(alvo)

features_novas = []
for coluna in novas.columns:
    valores = novas[coluna]
    features_novas.append({
        'coluna': coluna,
        'histograma': histograma(valores, 20),
        'assimetria': arredondar(valores.skew(), 2),
        'media': arredondar(valores.mean(), 3),
        'desvio': arredondar(valores.std(), 3),
        'minimo': arredondar(valores.min(), 3),
        'maximo': arredondar(valores.max(), 3),
        'por_classe': {
            'noventa_ou_mais': caixa(valores[alvo == 0]),
            'abaixo_de_90': caixa(valores[alvo == 1]),
        },
    })

correlacao_features = [
    {
        'coluna': c,
        'nova': c in novas.columns,
        'com_geral': arredondar(com_geral[c], 3),
        'com_abaixo_de_90': arredondar(com_abaixo[c], 3),
        # quatro casas para a tela arredondar uma vez só
        'com_geral_exata': arredondar(com_geral[c], 4),
        'com_abaixo_de_90_exata': arredondar(com_abaixo[c], 4),
    }
    for c in com_geral.abs().sort_values(ascending=False).index
]
# todo par com correlação de 0,8 ou mais, em módulo, entre notas e features novas
redundancia = comparacao.corr()
pares = [
    (a, b) for i, a in enumerate(redundancia.columns) for b in redundancia.columns[i + 1:]
    if abs(redundancia.loc[a, b]) >= 0.8
]
redundantes = sorted(
    ({'a': a, 'b': b, 'correlacao': arredondar(redundancia.loc[a, b], 2)} for a, b in pares),
    key=lambda par: -abs(par['correlacao']),
)

porte = np.log(origem[[c for c in origem.columns if c.startswith('Total avaliado')]].sum(axis=1))
descartadas = {
    'assimetria_ind4': arredondar(modeladas['ind4'].skew(), 2),
    'assimetria_log_ind4': arredondar(np.log(modeladas['ind4']).skew(), 2),
    'correlacao_porte_geral': arredondar(porte.corr(modeladas['geral']), 3),
    'subindicador_2_3_para_ind2': {
        str(k): {str(kk): int(vv) for kk, vv in linha.items()}
        for k, linha in pd.crosstab(origem['Desempenho — Subindicador 2.3'], modeladas['ind2']).iterrows()
    },
}

print(pd.DataFrame(correlacao_features))
print(pd.DataFrame(redundantes))
print(descartadas)

                coluna   nova  com_geral  com_abaixo_de_90  com_geral_exata  \
0          ind3_x_ind4   True      0.933            -0.338           0.9331   
1                 ind3  False      0.906            -0.311           0.9064   
2   desvio_indicadores   True      0.818            -0.174           0.8177   
3                 ind4  False      0.491            -0.235           0.4911   
4  indicadores_na_meta   True      0.408            -0.213           0.4077   
5     ind3_abaixo_meta   True     -0.373             0.208          -0.3730   
6                 ind1  False      0.369            -0.701           0.3694   
7            taxa_ind1   True      0.325            -0.629           0.3247   
8                 ind2  False      0.265            -0.175           0.2649   
9    media_blocos_ind4   True      0.113            -0.255           0.1125   

   com_abaixo_de_90_exata  
0                 -0.3377  
1                 -0.3106  
2                 -0.1742  
3                 

## Etapa 4: encoding e padronização

In [7]:
from sklearn.preprocessing import LabelEncoder

# a mesma codificação do caderno 02, só para medir o tamanho final da tabela
nivel = modeladas['tipo'].str.extract(r'\s(\d+)$')[0].fillna(0).astype(int)
codificada = pd.concat([modeladas, novas], axis=1)
codificada['tipo_codigo'] = LabelEncoder().fit_transform(codificada['tipo'])
codificada['nivel_tipo'] = nivel
codificada = pd.get_dummies(codificada, columns=['familia', 'distrito'], dtype=int)

mac = modeladas['familia'] == 'MAC'
nivel_mac = [{'nivel': int(n), **caixa(modeladas.loc[mac & (nivel == n), 'geral'])} for n in sorted(nivel[mac].unique())]

continuas = notas + ['taxa_ind1', 'ind3_x_ind4', 'desvio_indicadores', 'media_blocos_ind4']
escala = pd.concat([modeladas[notas], novas], axis=1)[continuas]

codificacao = {
    'one_hot': [
        {'coluna': 'familia', 'valores': int(modeladas['familia'].nunique())},
        {'coluna': 'distrito', 'valores': int(modeladas['distrito'].nunique())},
    ],
    'label_encoding': {'coluna': 'tipo', 'valores': int(modeladas['tipo'].nunique())},
    'ordinal': {'coluna': 'nivel_tipo', 'de': int(nivel.min()), 'ate': int(nivel.max())},
    'linhas': int(codificada.shape[0]),
    'colunas_finais': int(codificada.shape[1]),
    'nivel_mac': nivel_mac,
    'escala': [
        {'coluna': c, 'media': arredondar(escala[c].mean(), 3), 'desvio': arredondar(escala[c].std(), 3)}
        for c in continuas
    ],
}
print(json.dumps(codificacao, ensure_ascii=False, indent=1))

{
 "one_hot": [
  {
   "coluna": "familia",
   "valores": 6
  },
  {
   "coluna": "distrito",
   "valores": 8
  }
 ],
 "label_encoding": {
  "coluna": "tipo",
  "valores": 17
 },
 "ordinal": {
  "coluna": "nivel_tipo",
  "de": 0,
  "ate": 8
 },
 "linhas": 190,
 "colunas_finais": 28,
 "nivel_mac": [
  {
   "nivel": 1,
   "n": 10,
   "q1": 1.2184,
   "mediana": 1.2345,
   "q3": 1.2541,
   "bigode_baixo": 1.1973,
   "bigode_alto": 1.2703,
   "fora": []
  },
  {
   "nivel": 2,
   "n": 2,
   "q1": 1.2272,
   "mediana": 1.2399,
   "q3": 1.2525,
   "bigode_baixo": 1.2146,
   "bigode_alto": 1.2652,
   "fora": []
  },
  {
   "nivel": 3,
   "n": 2,
   "q1": 0.765,
   "mediana": 0.7776,
   "q3": 0.7902,
   "bigode_baixo": 0.7524,
   "bigode_alto": 0.8028,
   "fora": []
  },
  {
   "nivel": 4,
   "n": 5,
   "q1": 0.7491,
   "mediana": 0.7531,
   "q3": 0.776,
   "bigode_baixo": 0.7491,
   "bigode_alto": 0.776,
   "fora": [
    0.646,
    0.8364
   ]
  }
 ],
 "escala": [
  {
   "coluna": "ind1",
   

## Gravar

O arquivo leva a data e a origem, como `resultados.json`. Nenhum número aqui entra no cálculo
da gratificação, e nenhuma linha identifica pessoa: a unidade é a menor coisa que aparece.

In [8]:
saida = {
    'gerado_em': '2026-09-23',
    'fonte': 'ml/data/base nova completa.csv',
    'cadernos': ['01-eda', '02-preprocessamento'],
    'base': base,
    'classes': classes,
    'classes_196': classes_196,
    'assimetria_geral_modeladas': assimetria_geral_modeladas,
    'distribuicao': distribuicao,
    'por_familia': por_familia,
    'por_distrito': por_distrito,
    'correlacao': correlacao,
    'mais_correlatas': [
        {'coluna': c, 'r': arredondar(r, 4), 'r_modeladas': arredondar(nas_modeladas[c], 4)}
        for c, r in mais_correlatas.items()
    ],
    'faltantes': faltantes,
    'outliers': outliers,
    'outliers_geral_por_familia': {k: int(v) for k, v in outliers_geral_por_familia.items()},
    'inconsistencias': inconsistencias,
    'dispersao': dispersao,
    'linhas_fora_da_regra': linhas_fora,
    'features_novas': features_novas,
    'correlacao_features': correlacao_features,
    'redundantes': redundantes,
    'descartadas': descartadas,
    'codificacao': codificacao,
}

destino = Path('../../src/content/ml/apresentacao.json')
destino.write_text(json.dumps(saida, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('gravado:', destino.resolve().relative_to(Path('../..').resolve()), '|', destino.stat().st_size, 'bytes')

gravado: src/content/ml/apresentacao.json | 42992 bytes
